# ENDG 511 — Hair Analysis System: CNN Training Pipeline
**Team 1 | Darren Taylor · Naishah Adetunji · Sehba Samman**

This notebook implements the advanced goal training pipeline for the Smart Hair Analysis System.
It runs three stages:
- **Stage 1**: SimCLR self-supervised pretraining on 13,233 unlabelled LFW images
- **Stage 2**: MobileNetV2 supervised fine-tuning on 12,000 labelled CelebA images
- **Stage 3**: Prototypical few-shot domain adaptation

**Requirements**: Runtime must be set to T4 GPU (Runtime → Change runtime type → T4 GPU)

**GitHub**: https://github.com/choyacore/ENDG-511-final

In [ ]:
# Clone repo from GitHub
!git clone https://github.com/choyacore/ENDG-511-final.git
# Change into the project directory
%cd ENDG-511-final
# Install required dependencies
!pip install mediapipe scikit-learn torch torchvision -q

Cloning into 'ENDG-511-final'...
remote: Enumerating objects: 24, done.
remote: Counting objects: 100% (24/24), done.
remote: Compressing objects: 100% (22/22), done.
remote: Total 24 (delta 4), reused 19 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (24/24), 48.86 KiB | 862.00 KiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/ENDG-511-final
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 77.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 14.1 MB/s eta 0:00:00


## Setup: Mount Google Drive and Load Dataset

The dataset (~500MB) is stored on Google Drive as a compressed archive.
It contains:
- `black/`, `brown/`, `blonde/`, `gray/` — 3,000 CelebA images each (12,000 total, labelled)
- `unlabelled/` — 13,233 LFW images (no labels, used for SSL pretraining)



In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
!tar -xzf "/content/drive/MyDrive/endg 511/dataset_training.tar.gz"
# Verify the folders extracted correctly
!ls dataset/

Mounted at /content/drive
tar: Ignoring unknown extended header keyword 'SCHILY.fflags'
tar: Ignoring unknown extended header keyword 'SCHILY.fflags'
tar: Ignoring unknown extended header keyword 'SCHILY.fflags'
tar: Ignoring unknown extended header keyword 'SCHILY.fflags'
tar: Ignoring unknown extended header keyword 'SCHILY.fflags'
tar: Ignoring unknown extended header keyword 'SCHILY.fflags'
black  blonde  brown  gray  unlabelled


In [ ]:
#Import torch
import torch
#Check GPU
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
Tesla T4


## Stage 1 + 2 + 3: Full Training Pipeline

Runs all three stages sequentially:

**Stage 1 — SimCLR SSL pretraining (30 epochs)**
- Uses 13,233 unlabelled LFW images
- Learns lighting/texture invariance via contrastive learning
- No class labels needed
- Expected NT-Xent loss: starts ~1.9, should reach ~0.07 by epoch 30

**Stage 2 — MobileNetV2 fine-tuning (20 epochs)**
- Loads SSL backbone from Stage 1
- Trains on 12,000 CelebA images (9,600 train / 2,400 val)
- Classes: black, brown, blonde, gray
- Expected val accuracy: ~82-83%

**Stage 3 — Few-shot domain adaptation (50 episodes)**
- Uses trained classifier from Stage 2
- K=5 support images, 10 query images per class
- Note: uses CelebA proxy images (no real webcam data) — see Stage 3 cell for explanation


In [ ]:
!python train_cnn.py --task color --data ./dataset --ssl-epochs 30 --ft-epochs 20 --episodes 50


  Hair Analysis Trainer  |  task=color  |  device=cuda


  STAGE 1 -- SimCLR Self-Supervised Pretraining
[INFO] SSL pool (unlabelled/) : 13233 images
Downloading: "https://download.pytorch.org/models/mobilenet_v2-b0353104.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v2-b0353104.pth
100% 13.6M/13.6M [00:00<00:00, 138MB/s]
  SSL Epoch 001/30  |  NT-Xent = 1.9365  <- backbone saved
  SSL Epoch 002/30  |  NT-Xent = 0.8487  <- backbone saved
  SSL Epoch 003/30  |  NT-Xent = 0.5813  <- backbone saved
  SSL Epoch 004/30  |  NT-Xent = 0.4699  <- backbone saved
  SSL Epoch 005/30  |  NT-Xent = 0.3948  <- backbone saved
  SSL Epoch 006/30  |  NT-Xent = 0.3499  <- backbone saved
  SSL Epoch 007/30  |  NT-Xent = 0.3224  <- backbone saved
  SSL Epoch 008/30  |  NT-Xent = 0.3102  <- backbone saved
  SSL Epoch 009/30  |  NT-Xent = 0.2720  <- backbone saved
  SSL Epoch 010/30  |  NT-Xent = 0.2514  <- backbone saved
  SSL Epoch 011/30  |  NT-Xent = 0.2383  <- backbone saved
  SSL Epoch 012/30 

In [ ]:
import shutil
shutil.copy('hair_color_classifier.pt', '/content/drive/MyDrive/endg 511/hair_color_classifier_full.pt')
shutil.copy('ssl_backbone_color.pt', '/content/drive/MyDrive/endg 511/ssl_backbone_color.pt')
print("Weights saved!")

Why 7 classes but only 4 trained?
Our code defines 7 hair colour classes (black, brown, blonde, red, auburn, gray, white) but CelebA only had labelled data for 4 of them. Stage 1 SimCLR doesn't care about classes at all — it just takes all 13,233 LFW images and learns visual features by comparing augmented versions of the same image. No labels needed, which is why it ran fine.
For Stage 2 fine-tuning, the model reserved 7 output slots but red, auburn, and white never received any training examples — their weights just stayed at random initialization. The model effectively learned to distinguish between the 4 classes it actually saw.
For the subgroup evaluation we masked the output to only look at the 4 valid class indices (black=0, brown=1, blonde=2, gray=5) so the report reflects actual performance rather than being confused by the empty class slots.

Stage 3

In [ ]:
import os
import shutil
import random

# Create webcam support folder
os.makedirs('./webcam_support/black', exist_ok=True)
os.makedirs('./webcam_support/brown', exist_ok=True)
os.makedirs('./webcam_support/blonde', exist_ok=True)
os.makedirs('./webcam_support/gray', exist_ok=True)

# Copy 20 random images from each CelebA class as "webcam support"
for color in ['black', 'brown', 'blonde', 'gray']:
    source = f'./dataset/{color}'
    webcam = f'./webcam_support/{color}'
    files = os.listdir(source)
    selected = random.sample(files, 20)
    for f in selected:
        shutil.copy(os.path.join(source, f), os.path.join(webcam, f))
    print(f'{color}: copied 20 images')

print("Done!")

black: copied 20 images
brown: copied 20 images
blonde: copied 20 images
gray: copied 20 images
Done!


In [ ]:
!python train_cnn.py --task color --data ./webcam_support --few-shot-only --episodes 50


  Hair Analysis Trainer  |  task=color  |  device=cuda

[INFO] Loaded color classifier: hair_color_proto_adapted.pt
  STAGE 3 -- Prototypical Network Few-Shot Adaptation
  K-shot=5  Q-query=10  Episodes=50

[INFO] Labelled dataset : 159 images | 4 classes
  Episode 001/50  |  ep_acc=0.025  |  running_mean=0.025
  Episode 010/50  |  ep_acc=0.025  |  running_mean=0.075
  Episode 020/50  |  ep_acc=0.050  |  running_mean=0.070
  Episode 030/50  |  ep_acc=0.050  |  running_mean=0.073
  Episode 040/50  |  ep_acc=0.125  |  running_mean=0.079
  Episode 050/50  |  ep_acc=0.050  |  running_mean=0.074

[FEW-SHOT DONE]
  Mean acc = 0.074 +/- 0.011  (95% CI)
  Best acc = 0.175  |  Model -> hair_color_proto_adapted.pt

  Subgroup Evaluation (Per-Class Metrics)
[INFO] Labelled dataset : 159 images | 4 classes

  Classification Report:
              precision    recall  f1-score   support

       black      0.314     0.275     0.293        40
       brown      0.258     0.400     0.314        40
    

In [ ]:

import shutil
# Save the few-shot adapted model
shutil.copy('hair_color_proto_adapted.pt', '/content/drive/MyDrive/endg 511/hair_color_proto_adapted.pt')
# Save the fine-tuned classifier (best Stage 2 model)
shutil.copy('hair_color_classifier.pt', '/content/drive/MyDrive/endg 511/hair_color_classifier_4class.pt')
print("All saved!")

All saved!


In [ ]:
from train_cnn import HairClassifier, HairDataset, get_val_transform
from sklearn.metrics import classification_report, confusion_matrix
from torch.utils.data import DataLoader
import torch
import numpy as np

device = torch.device('cuda')
classes_4 = ['black', 'brown', 'blonde', 'gray']

hair_model = HairClassifier(4)
hair_model.load_state_dict(torch.load('/content/drive/MyDrive/endg 511/hair_color_classifier_full.pt', map_location=device))
hair_model = hair_model.to(device)
hair_model.eval()

ds = HairDataset('/content/ENDG-511-final/dataset', classes_4, transform=get_val_transform())
dl = DataLoader(ds, batch_size=32, shuffle=False)
# Run inference — no gradients needed for evaluation
all_prediction, all_labels = [], []
with torch.no_grad():
    for imgs, labels in dl:
        preds = hair_model(imgs.to(device)).argmax(1).cpu() # get predicted class index
        all_prediction.extend(preds.numpy())
        all_labels.extend(labels.numpy())

all_prediction = np.array(all_prediction)
all_labels = np.array(all_labels)
# Print per-class precision, recall, F1, and confusion matrix
print(classification_report(all_labels, all_prediction, target_names=classes_4, digits=3))
print("Confusion Matrix:")
print(confusion_matrix(all_labels, all_prediction))

[INFO] Labelled dataset : 12000 images | 4 classes
              precision    recall  f1-score   support

       black      0.832     0.896     0.862      3000
       brown      0.816     0.742     0.777      3000
      blonde      0.878     0.882     0.880      3000
        gray      0.895     0.903     0.899      3000

    accuracy                          0.856     12000
   macro avg      0.855     0.856     0.855     12000
weighted avg      0.855     0.856     0.855     12000

Confusion Matrix:
[[2687  274    2   37]
 [ 459 2227  201  113]
 [  18  169 2645  168]
 [  67   59  164 2710]]
